# Retail Decision Intelligence - Data Audit

Exploratory audit of two source datasets:
1. `retail_sales_ml_apl.csv` - Sales transactions
2. `retail_inventory_ml_apl.csv` - Inventory snapshots

**Rule**: Raw data is immutable. No modifications, cleaning, or transformations.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../data/raw")
sales_path = RAW_DIR / "retail_sales_ml_apl.csv"
inv_path = RAW_DIR / "retail_inventory_ml_apl.csv"

print(f"Sales file size: {sales_path.stat().st_size / 1e6:.2f} MB")
print(f"Inventory file size: {inv_path.stat().st_size / 1e6:.2f} MB")

Sales file size: 39.07 MB
Inventory file size: 99.99 MB


In [2]:
# Load data with appropriate dtypes
sales = pd.read_csv(sales_path, low_memory=False)
inv = pd.read_csv(inv_path, low_memory=False)

print(f"Sales shape: {sales.shape}")
print(f"Inventory shape: {inv.shape}")

print("\nSales columns:")
print(sales.columns.tolist())
print("\nInventory columns:")
print(inv.columns.tolist())

Sales shape: (125751, 17)
Inventory shape: (284755, 18)

Sales columns:
['Transaction Date', 'Sales Type', 'Is Return', 'Reason of Return', 'Supplier', 'Product No', 'Product Description', 'Product Division', 'Product Category', 'Product Subcategory', 'Product Segment', 'Store', 'Sales Channel', 'Qty Sold', 'Sales Amount', 'Cogs', 'Number of Transactions']

Inventory columns:
['Start Date', 'End Date', 'Stock Status', 'Supplier', 'Product No', 'Product Description', 'Product Division', 'Product Category', 'Product Subcategory', 'Product Segment', 'Store', 'Store Type', 'Sales Channel', 'Qty on hand', 'Stocks Selling Amount', 'Cost of Stocks', 'Stock Unit Selling Price', 'Stock Unit Cost Price']


In [3]:
# Basic info
print("=== SALES ===")
sales.info()
print("\n=== INVENTORY ===")
inv.info()

=== SALES ===
<class 'pandas.DataFrame'>
RangeIndex: 125751 entries, 0 to 125750
Data columns (total 17 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Transaction Date        125751 non-null  str    
 1   Sales Type              125751 non-null  str    
 2   Is Return               125751 non-null  int64  
 3   Reason of Return        0 non-null       float64
 4   Supplier                125751 non-null  str    
 5   Product No              125751 non-null  str    
 6   Product Description     125751 non-null  str    
 7   Product Division        125751 non-null  str    
 8   Product Category        125751 non-null  str    
 9   Product Subcategory     125751 non-null  str    
 10  Product Segment         125751 non-null  str    
 11  Store                   125751 non-null  str    
 12  Sales Channel           125751 non-null  str    
 13  Qty Sold                125751 non-null  float64
 14  Sales Amount     

<class 'pandas.DataFrame'>
RangeIndex: 284755 entries, 0 to 284754
Data columns (total 18 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Start Date                284755 non-null  str    
 1   End Date                  284755 non-null  str    
 2   Stock Status              284755 non-null  str    
 3   Supplier                  284755 non-null  str    
 4   Product No                284755 non-null  str    
 5   Product Description       284755 non-null  str    
 6   Product Division          284755 non-null  str    
 7   Product Category          284755 non-null  str    
 8   Product Subcategory       284755 non-null  str    
 9   Product Segment           284755 non-null  str    
 10  Store                     284755 non-null  str    
 11  Store Type                284755 non-null  str    
 12  Sales Channel             284755 non-null  str    
 13  Qty on hand               284755 non-null  int64  
 14 

In [4]:
# Date ranges
sales['Transaction Date'] = pd.to_datetime(sales['Transaction Date'])
inv['Start Date'] = pd.to_datetime(inv['Start Date'])
inv['End Date'] = pd.to_datetime(inv['End Date'])

print(f"Sales date range: {sales['Transaction Date'].min()} to {sales['Transaction Date'].max()}")
print(f"Inventory Start date range: {inv['Start Date'].min()} to {inv['Start Date'].max()}")
print(f"Inventory End date range: {inv['End Date'].min()} to {inv['End Date'].max()}")

Sales date range: 2025-06-01 00:00:00 to 2026-04-24 00:00:00
Inventory Start date range: 2025-06-01 00:00:00 to 2026-04-24 00:00:00
Inventory End date range: 2025-06-01 00:00:00 to 9999-12-31 00:00:00


In [5]:
# Unique counts
print("=== UNIQUE COUNTS ===")
print(f"Unique stores (sales): {sales['Store'].nunique()}")
print(f"Unique stores (inv): {inv['Store'].nunique()}")
print(f"Unique products/SKUs (sales): {sales['Product No'].nunique()}")
print(f"Unique products/SKUs (inv): {inv['Product No'].nunique()}")
print(f"Unique suppliers (sales): {sales['Supplier'].nunique()}")
print(f"Unique suppliers (inv): {inv['Supplier'].nunique()}")
print(f"Product Divisions (sales): {sales['Product Division'].nunique()}")
print(f"Product Categories (sales): {sales['Product Category'].nunique()}")
print(f"Product Subcategories (sales): {sales['Product Subcategory'].nunique()}")
print(f"Product Segments (sales): {sales['Product Segment'].nunique()}")

=== UNIQUE COUNTS ===
Unique stores (sales): 40
Unique stores (inv): 40


Unique products/SKUs (sales): 2326
Unique products/SKUs (inv): 2326
Unique suppliers (sales): 30
Unique suppliers (inv): 30
Product Divisions (sales): 3
Product Categories (sales): 27
Product Subcategories (sales): 51
Product Segments (sales): 600


In [6]:
# Duplicate records
print(f"Sales duplicate rows: {sales.duplicated().sum()}")
print(f"Inventory duplicate rows: {inv.duplicated().sum()}")

# Check for duplicate keys
sales_key_dupes = sales.duplicated(subset=['Transaction Date', 'Store', 'Product No']).sum()
inv_key_dupes = inv.duplicated(subset=['Start Date', 'End Date', 'Store', 'Product No']).sum()
print(f"Sales duplicate (date, store, product): {sales_key_dupes}")
print(f"Inventory duplicate (start, end, store, product): {inv_key_dupes}")

Sales duplicate rows: 0


Inventory duplicate rows: 0
Sales duplicate (date, store, product): 1823
Inventory duplicate (start, end, store, product): 0


In [7]:
# Missing values
print("=== SALES MISSING ===")
print(sales.isnull().sum())
print("\n=== INVENTORY MISSING ===")
print(inv.isnull().sum())

=== SALES MISSING ===


Transaction Date               0
Sales Type                     0
Is Return                      0
Reason of Return          125751
Supplier                       0
Product No                     0
Product Description            0
Product Division               0
Product Category               0
Product Subcategory            0
Product Segment                0
Store                          0
Sales Channel                  0
Qty Sold                       0
Sales Amount                   0
Cogs                           0
Number of Transactions         0
dtype: int64

=== INVENTORY MISSING ===


Start Date                  0
End Date                    0
Stock Status                0
Supplier                    0
Product No                  0
Product Description         0
Product Division            0
Product Category            0
Product Subcategory         0
Product Segment             0
Store                       0
Store Type                  0
Sales Channel               0
Qty on hand                 0
Stocks Selling Amount       0
Cost of Stocks              0
Stock Unit Selling Price    0
Stock Unit Cost Price       0
dtype: int64


In [8]:
# Numerical field analysis - Sales
num_cols_sales = ['Qty Sold', 'Sales Amount', 'Cogs', 'Number of Transactions']
for col in num_cols_sales:
    s = sales[col]
    print(f"\n{col}:")
    print(f"  Min: {s.min()}, Max: {s.max()}, Mean: {s.mean():.2f}, Median: {s.median()}")
    print(f"  Zeros: {(s == 0).sum()}, Negatives: {(s < 0).sum()}")
    print(f"  Missing: {s.isnull().sum()}")


Qty Sold:
  Min: -3.0, Max: 14.0, Mean: 0.93, Median: 1.0
  Zeros: 2, Negatives: 7491
  Missing: 0

Sales Amount:
  Min: -380.0, Max: 2660.0, Mean: 83.36, Median: 90.0
  Zeros: 9, Negatives: 7491
  Missing: 0

Cogs:
  Min: -192.28, Max: 1345.96, Mean: 46.50, Median: 50.05
  Zeros: 2, Negatives: 7491
  Missing: 0

Number of Transactions:
  Min: 1, Max: 14, Mean: 1.04, Median: 1.0
  Zeros: 0, Negatives: 0
  Missing: 0


In [9]:
# Numerical field analysis - Inventory
num_cols_inv = ['Qty on hand', 'Stocks Selling Amount', 'Cost of Stocks', 'Stock Unit Selling Price', 'Stock Unit Cost Price']
for col in num_cols_inv:
    s = inv[col]
    print(f"\n{col}:")
    print(f"  Min: {s.min()}, Max: {s.max()}, Mean: {s.mean():.2f}, Median: {s.median()}")
    print(f"  Zeros: {(s == 0).sum()}, Negatives: {(s < 0).sum()}")
    print(f"  Missing: {s.isnull().sum()}")


Qty on hand:
  Min: -9, Max: 56, Mean: 2.29, Median: 2.0
  Zeros: 13869, Negatives: 1380
  Missing: 0

Stocks Selling Amount:
  Min: -828.0, Max: 5600.0, Mean: 224.39, Median: 160.0
  Zeros: 13869, Negatives: 1380
  Missing: 0

Cost of Stocks:
  Min: -418.95, Max: 2641.45, Mean: 111.05, Median: 80.96
  Zeros: 362, Negatives: 8022
  Missing: 0

Stock Unit Selling Price:
  Min: 0.0, Max: 250.0, Mean: 95.31, Median: 100.0
  Zeros: 13869, Negatives: 0
  Missing: 0

Stock Unit Cost Price:
  Min: -16.94, Max: 336.38, Mean: 47.42, Median: 50.05
  Zeros: 13869, Negatives: 25
  Missing: 0


In [10]:
# Sales Type distribution
print("Sales Type:")
print(sales['Sales Type'].value_counts())
print(f"\nIs Return:")
print(sales['Is Return'].value_counts())
print(f"\nReason of Return (non-null):")
print(sales['Reason of Return'].dropna().value_counts())

Sales Type:
Sales Type
Full Price         93929
Promo              23651
Markdown Tier 1     4384
Markdown Tier 2     2434
Clearance           1353
Name: count, dtype: int64

Is Return:
Is Return
0    118260
1      7491
Name: count, dtype: int64

Reason of Return (non-null):
Series([], Name: count, dtype: int64)


In [11]:
# Returns analysis
returns = sales[sales['Is Return'] == 1]
print(f"Return transactions: {len(returns)}")
if len(returns) > 0:
    print(f"Return Qty Sold stats:")
    print(returns['Qty Sold'].describe())
    print(f"Return Sales Amount stats:")
    print(returns['Sales Amount'].describe())

Return transactions: 7491
Return Qty Sold stats:
count    7491.000000
mean       -1.007476
std         0.089190
min        -3.000000
25%        -1.000000
50%        -1.000000
75%        -1.000000
max        -1.000000
Name: Qty Sold, dtype: float64
Return Sales Amount stats:
count    7491.000000
mean      -97.926980
std        40.667809
min      -380.000000
25%      -120.000000
50%       -95.000000
75%       -69.990000
max        -4.990000
Name: Sales Amount, dtype: float64


In [12]:
# Stock Status distribution
print("Stock Status:")
print(inv['Stock Status'].value_counts())
print(f"\nStore Type:")
print(inv['Store Type'].value_counts())
print(f"\nSales Channel:")
print(inv['Sales Channel'].value_counts())

Stock Status:


Stock Status
Full Price         261899
Markdown Tier 1     12091
Markdown Tier 2      6937
Clearance            3798
Promo                  30
Name: count, dtype: int64

Store Type:
Store Type
Retail Storefront    284755
Name: count, dtype: int64

Sales Channel:


Sales Channel
Channel Alpha    284755
Name: count, dtype: int64


In [13]:
# Inventory duration analysis
inv['Duration Days'] = (inv['End Date'] - inv['Start Date']).dt.days
print(f"Inventory duration stats:")
print(inv['Duration Days'].describe())

# Check for same-day start/end (potential stockout snapshots)
same_day = inv[inv['Duration Days'] == 0]
print(f"\nSame-day start/end records: {len(same_day)} ({len(same_day)/len(inv)*100:.1f}%)")

Inventory duration stats:


count    2.847550e+05
mean     2.322062e+05
std      7.888516e+05
min      0.000000e+00
25%      2.000000e+00
50%      9.000000e+00
75%      4.900000e+01
max      2.912656e+06
Name: Duration Days, dtype: float64

Same-day start/end records: 34998 (12.3%)


In [14]:
# Can inventory be reconstructed at daily product-store level?
# Check if inventory records are continuous or snapshots
sample_product = inv['Product No'].iloc[0]
sample_store = inv['Store'].iloc[0]
subset = inv[(inv['Product No'] == sample_product) & (inv['Store'] == sample_store)].sort_values('Start Date')
print(f"Sample product-store inventory records: {len(subset)}")
print(subset[['Start Date', 'End Date', 'Qty on hand', 'Stock Status']].head(10))

# Check for gaps
subset['Gap Days'] = subset['Start Date'].diff().dt.days
print(f"\nGaps between records (days):")
print(subset['Gap Days'].value_counts().head())

Sample product-store inventory records: 14


       Start Date   End Date  Qty on hand Stock Status
82477  2025-06-22 2025-06-23            2   Full Price
77217  2025-06-24 2025-07-05            2   Full Price
77641  2025-07-06 2025-07-22            1   Full Price
0      2025-08-07 2025-10-18            1   Full Price
143369 2025-10-19 2025-10-20            2   Full Price
152576 2025-10-21 2025-11-08            2   Full Price
192963 2025-11-09 2025-11-10            4   Full Price
281677 2025-11-11 2026-03-10            4   Full Price
105525 2026-03-11 2026-03-20            3   Full Price
164221 2026-03-21 2026-04-03            2   Full Price

Gaps between records (days):
Gap Days
2.0     4
12.0    1
32.0    1
73.0    1
19.0    1
Name: count, dtype: int64


In [15]:
# Sales-Inventory relationship
# Check overlap in products and stores
sales_products = set(sales['Product No'].unique())
inv_products = set(inv['Product No'].unique())
sales_stores = set(sales['Store'].unique())
inv_stores = set(inv['Store'].unique())

print(f"Products in sales only: {len(sales_products - inv_products)}")
print(f"Products in inventory only: {len(inv_products - sales_products)}")
print(f"Products in both: {len(sales_products & inv_products)}")
print(f"\nStores in sales only: {len(sales_stores - inv_stores)}")
print(f"Stores in inventory only: {len(inv_stores - sales_stores)}")
print(f"Stores in both: {len(sales_stores & inv_stores)}")

Products in sales only: 0
Products in inventory only: 0
Products in both: 2326

Stores in sales only: 0
Stores in inventory only: 0
Stores in both: 40


In [16]:
# Price, cost, discount, promotion, markdown fields
print("=== PRICE/COST ANALYSIS ===")
print(f"Sales: Unit price implied = Sales Amount / Qty Sold")
sales['Unit Price'] = sales['Sales Amount'] / sales['Qty Sold']
sales['Unit Cost'] = sales['Cogs'] / sales['Qty Sold']
sales['Margin'] = sales['Unit Price'] - sales['Unit Cost']
sales['Margin Pct'] = sales['Margin'] / sales['Unit Price']

print(f"Sales Unit Price: min={sales['Unit Price'].min():.2f}, max={sales['Unit Price'].max():.2f}, mean={sales['Unit Price'].mean():.2f}")
print(f"Sales Unit Cost: min={sales['Unit Cost'].min():.2f}, max={sales['Unit Cost'].max():.2f}, mean={sales['Unit Cost'].mean():.2f}")
print(f"Sales Margin Pct: min={sales['Margin Pct'].min():.4f}, max={sales['Margin Pct'].max():.4f}, mean={sales['Margin Pct'].mean():.4f}")

print(f"\nInventory Unit Selling Price: min={inv['Stock Unit Selling Price'].min():.2f}, max={inv['Stock Unit Selling Price'].max():.2f}, mean={inv['Stock Unit Selling Price'].mean():.2f}")
print(f"Inventory Unit Cost Price: min={inv['Stock Unit Cost Price'].min():.2f}, max={inv['Stock Unit Cost Price'].max():.2f}, mean={inv['Stock Unit Cost Price'].mean():.2f}")

=== PRICE/COST ANALYSIS ===
Sales: Unit price implied = Sales Amount / Qty Sold
Sales Unit Price: min=0.00, max=250.00, mean=89.05
Sales Unit Cost: min=1.43, max=172.73, mean=49.81
Sales Margin Pct: min=-inf, max=0.9643, mean=-inf

Inventory Unit Selling Price: min=0.00, max=250.00, mean=95.31
Inventory Unit Cost Price: min=-16.94, max=336.38, mean=47.42


In [17]:
# Promotion analysis
print("Sales Type breakdown:")
promo_sales = sales[sales['Sales Type'] == 'Promo']
full_price_sales = sales[sales['Sales Type'] == 'Full Price']
print(f"Promo transactions: {len(promo_sales)} ({len(promo_sales)/len(sales)*100:.1f}%)")
print(f"Full Price transactions: {len(full_price_sales)} ({len(full_price_sales)/len(sales)*100:.1f}%)")

print(f"\nPromo avg unit price: {promo_sales['Unit Price'].mean():.2f}")
print(f"Full Price avg unit price: {full_price_sales['Unit Price'].mean():.2f}")

# Check for markdown/discount fields - none explicit, but Sales Type indicates promo

Sales Type breakdown:
Promo transactions: 23651 (18.8%)
Full Price transactions: 93929 (74.7%)

Promo avg unit price: 61.23
Full Price avg unit price: 100.65


In [18]:
# Suspicious patterns
print("=== SUSPICIOUS PATTERNS ===")

# 1. Negative or zero sales amount with positive qty
suspect1 = sales[(sales['Sales Amount'] <= 0) & (sales['Qty Sold'] > 0)]
print(f"Zero/negative sales amount with positive qty: {len(suspect1)}")

# 2. Positive sales amount with zero/negative qty
suspect2 = sales[(sales['Sales Amount'] > 0) & (sales['Qty Sold'] <= 0)]
print(f"Positive sales amount with zero/negative qty: {len(suspect2)}")

# 3. COGS > Sales Amount (loss-making transactions)
suspect3 = sales[sales['Cogs'] > sales['Sales Amount']]
print(f"COGS > Sales Amount: {len(suspect3)} ({len(suspect3)/len(sales)*100:.2f}%)")

# 4. Inventory with zero qty on hand but positive selling amount
suspect4 = inv[(inv['Qty on hand'] == 0) & (inv['Stocks Selling Amount'] > 0)]
print(f"Inv zero qty but positive selling amount: {len(suspect4)}")

# 5. Inventory unit price mismatch
inv['Implied Unit Price'] = inv['Stocks Selling Amount'] / inv['Qty on hand']
price_mismatch = inv[abs(inv['Implied Unit Price'] - inv['Stock Unit Selling Price']) > 0.01]
print(f"Inventory unit price mismatch: {len(price_mismatch)}")

# 6. Future dates in sales
future_sales = sales[sales['Transaction Date'] > pd.Timestamp('2025-12-31')]
print(f"Sales with future dates (>2025): {len(future_sales)}")

=== SUSPICIOUS PATTERNS ===
Zero/negative sales amount with positive qty: 7
Positive sales amount with zero/negative qty: 0
COGS > Sales Amount: 14194 (11.29%)
Inv zero qty but positive selling amount: 0
Inventory unit price mismatch: 0
Sales with future dates (>2025): 38621


In [19]:
# Distribution analysis - Sales Quantity
print("=== SALES QUANTITY DISTRIBUTION ===")
print(sales['Qty Sold'].describe())
print(f"\nValue counts:")
print(sales['Qty Sold'].value_counts().head(20))

=== SALES QUANTITY DISTRIBUTION ===
count    125751.000000
mean          0.930370
std           0.586897
min          -3.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          14.000000
Name: Qty Sold, dtype: float64

Value counts:
Qty Sold
 1.0     113712
-1.0       7437
 2.0       3664
 3.0        491
 4.0        200
 5.0         80
-2.0         52
 6.0         40
 7.0         29
 8.0         21
 10.0         7
 9.0          7
 11.0         3
 0.0          2
 13.0         2
-3.0          2
 14.0         1
 12.0         1
Name: count, dtype: int64


In [20]:
# Distribution analysis - Revenue/Sales Amount
print("=== SALES AMOUNT DISTRIBUTION ===")
print(sales['Sales Amount'].describe())
print(f"\nPercentiles:")
print(sales['Sales Amount'].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

=== SALES AMOUNT DISTRIBUTION ===
count    125751.000000
mean         83.356079
std          83.658776
min        -380.000000
25%          50.000000
50%          90.000000
75%         115.000000
max        2660.000000
Name: Sales Amount, dtype: float64

Percentiles:
0.01   -130.00
0.05    -59.99
0.25     50.00
0.50     90.00
0.75    115.00
0.95    165.00
0.99    320.00
Name: Sales Amount, dtype: float64


In [21]:
# Distribution analysis - Inventory Quantity
print("=== INVENTORY QTY ON HAND DISTRIBUTION ===")
print(inv['Qty on hand'].describe())
print(f"\nValue counts:")
print(inv['Qty on hand'].value_counts().head(20))

=== INVENTORY QTY ON HAND DISTRIBUTION ===
count    284755.000000
mean          2.294629
std           2.007395
min          -9.000000
25%           1.000000
50%           2.000000
75%           3.000000
max          56.000000
Name: Qty on hand, dtype: float64

Value counts:
Qty on hand
 1     101217
 2      83347
 3      33954
 4      23181
 0      13869
 5      10282
 6       6395
 7       3333
 8       2393
 9       1695
 10      1163
-1       1129
 11       795
 12       605
 13       381
 14       240
-2        166
 15       163
 16       132
 17        85
Name: count, dtype: int64


In [22]:
# Stockout analysis
print("=== STOCKOUT ANALYSIS ===")

# Records with zero inventory
zero_inv = inv[inv['Qty on hand'] == 0]
print(f"Zero inventory records: {len(zero_inv)} ({len(zero_inv)/len(inv)*100:.1f}%)")

# Stock Status for zero qty
if len(zero_inv) > 0:
    print(f"Stock Status for zero qty:")
    print(zero_inv['Stock Status'].value_counts())

# Check if 'Stockout' status exists
print(f"\nAll Stock Status values:")
print(inv['Stock Status'].unique())

=== STOCKOUT ANALYSIS ===
Zero inventory records: 13869 (4.9%)
Stock Status for zero qty:
Stock Status
Full Price         12653
Markdown Tier 2      593
Markdown Tier 1      572
Clearance             51
Name: count, dtype: int64

All Stock Status values:
<StringArray>
['Full Price', 'Markdown Tier 1', 'Markdown Tier 2', 'Clearance', 'Promo']
Length: 5, dtype: str


In [23]:
# Product hierarchy analysis
print("=== PRODUCT HIERARCHY ===")
hierarchy = sales[['Product Division', 'Product Category', 'Product Subcategory', 'Product Segment']].drop_duplicates()
print(f"Total unique hierarchy combinations: {len(hierarchy)}")

print(f"\nDivisions:")
print(sales['Product Division'].value_counts())

print(f"\nCategories per Division:")
for div in sales['Product Division'].unique():
    cats = sales[sales['Product Division'] == div]['Product Category'].nunique()
    print(f"  {div}: {cats} categories")

=== PRODUCT HIERARCHY ===


Total unique hierarchy combinations: 600

Divisions:
Product Division
Scholar Footwear    76907
Femme Footwear      29848
Junior Apparel      18996
Name: count, dtype: int64

Categories per Division:


  Scholar Footwear: 10 categories
  Femme Footwear: 6 categories
  Junior Apparel: 11 categories


In [24]:
# Temporal coverage - sales by month
sales['YearMonth'] = sales['Transaction Date'].dt.to_period('M')
monthly_sales = sales.groupby('YearMonth').agg(
    transactions=('Transaction Date', 'count'),
    total_qty=('Qty Sold', 'sum'),
    total_revenue=('Sales Amount', 'sum'),
    unique_products=('Product No', 'nunique'),
    unique_stores=('Store', 'nunique')
)
print(monthly_sales)

           transactions  total_qty  total_revenue  unique_products  \
YearMonth                                                            
2025-06            8328     7712.0      706683.44             1056   
2025-07           10120     9545.0      855571.55             1291   
2025-08           18517    17561.0     1535117.21             1573   
2025-09           10499     9654.0      860457.23             1594   
2025-10            9750     9226.0      821266.08             1655   
2025-11           10118     9527.0      871946.34             1737   
2025-12           19798    18717.0     1785534.95             1909   
2026-01            7955     6609.0      565272.83             1674   
2026-02           10944    10222.0     1038967.86             1774   
2026-03           12902    11882.0      934194.48             1806   
2026-04            6820     6340.0      507098.28             1431   

           unique_stores  
YearMonth                 
2025-06               40  
2025-07 

In [25]:
# Temporal coverage - inventory by month
inv['Start YearMonth'] = inv['Start Date'].dt.to_period('M')
monthly_inv = inv.groupby('Start YearMonth').agg(
    records=('Start Date', 'count'),
    unique_products=('Product No', 'nunique'),
    unique_stores=('Store', 'nunique'),
    total_qty=('Qty on hand', 'sum')
)
print(monthly_inv)

                 records  unique_products  unique_stores  total_qty
Start YearMonth                                                    
2025-06            23063             1146             40      53261
2025-07            33617             1374             40      75231
2025-08            37639             1629             40      77605
2025-09            22312             1604             40      47033
2025-10            34982             1739             40      76863
2025-11            29268             1714             40      80106
2025-12            26428             1911             40      64228
2026-01            27504             1940             40      66078
2026-02            19242             1782             40      41692
2026-03            20105             1818             40      46885
2026-04            10595             1394             40      24425


In [26]:
# Supplier analysis
print("=== SUPPLIER ANALYSIS ===")
supplier_sales = sales.groupby('Supplier').agg(
    transactions=('Transaction Date', 'count'),
    revenue=('Sales Amount', 'sum'),
    products=('Product No', 'nunique'),
    stores=('Store', 'nunique')
).sort_values('revenue', ascending=False)
print(supplier_sales.head(20))

=== SUPPLIER ANALYSIS ===


             transactions     revenue  products  stores
Supplier                                               
Vendor 0166         42222  3290375.40       716      40
Vendor 0134         24567  3204043.60       391      40
Vendor 0169         20218  1703035.19       309      40
Vendor 0246          5017   537752.65        24      40
Vendor 0010          7488   490924.49       134      40
Vendor 0064          4278   423710.82        85      37
Vendor 0258          3587   258547.61        45      40
Vendor 0106          7362   175354.18       301      24
Vendor 0262          1290   111485.28        48      40
Vendor 0054          1928    77121.67        10      40
Vendor 0008           530    45562.33        21      21
Vendor 0037          1161    28432.74        59      21
Vendor 0255           533    25739.77         6      28
Vendor 0198          1872    20212.24        43      24
Vendor 0052           236    13340.49         4      40
Vendor 0193           220    12581.82         4 

In [27]:
# Store analysis
print("=== STORE ANALYSIS ===")
store_sales = sales.groupby('Store').agg(
    transactions=('Transaction Date', 'count'),
    revenue=('Sales Amount', 'sum'),
    products=('Product No', 'nunique')
).sort_values('revenue', ascending=False)
print(store_sales.head(20))

=== STORE ANALYSIS ===
          transactions    revenue  products
Store                                      
STR-1006          5904  485561.44      1812
STR-1229          3994  399696.48      1295
STR-1080          3639  371277.03      1192
STR-1194          4323  346391.53      1652
STR-1010          4916  343436.84      1778
STR-1235          3186  335338.71      1152
STR-1148          4705  332396.05      1728
STR-1094          4300  332104.96      1603
STR-1222          3446  329188.10      1388
STR-1044          4179  328466.36      1663
STR-1075          3947  327931.80      1529
STR-1198          4995  324919.24      1766
STR-1253          3586  303166.58      1506
STR-1109          3942  297203.61      1623
STR-1203          2934  284126.86      1179
STR-1114          3826  280115.59      1653
STR-1107          3710  274414.33      1581
STR-1163          2764  272628.13      1041
STR-1157          3747  270730.42      1562
STR-1145          3904  267684.43      1557


In [28]:
# Product-level sales summary
print("=== TOP PRODUCTS BY REVENUE ===")
product_sales = sales.groupby(['Product No', 'Product Description', 'Product Division', 'Product Category']).agg(
    total_qty=('Qty Sold', 'sum'),
    total_revenue=('Sales Amount', 'sum'),
    transactions=('Transaction Date', 'count'),
    stores=('Store', 'nunique'),
    avg_unit_price=('Unit Price', 'mean')
).sort_values('total_revenue', ascending=False)
print(product_sales.head(20))

=== TOP PRODUCTS BY REVENUE ===


                                                                                                                  total_qty  \
Product No  Product Description                                Product Division Product Category                              
PROD-140871 PROD-100439 SEG-500064 Scholar Footwear Boot Co... Scholar Footwear Scholar Footwear Boot Collection      915.0   
PROD-100805 PROD-102578 SEG-500351 Scholar Footwear Flight ... Scholar Footwear Scholar Footwear Flight Classic      1170.0   
PROD-100804 PROD-102576 SEG-500351 Scholar Footwear Flight ... Scholar Footwear Scholar Footwear Flight Classic       834.0   
PROD-100803 PROD-102577 SEG-500351 Scholar Footwear Flight ... Scholar Footwear Scholar Footwear Flight Classic       816.0   
PROD-140870 PROD-100438 SEG-500064 Scholar Footwear Boot Co... Scholar Footwear Scholar Footwear Boot Collection      546.0   
PROD-175338 PROD-113366 SEG-502458 Heritage Footwear Retro ... Scholar Footwear Scholar Footwear Retro Legend  